In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 0. Plotting setup
# ==========================================
# Use the seaborn default style for nicer looks
plt.style.use('seaborn-v0_8') 
# Chinese font fallback (uncomment if SimHei is installed)
# plt.rcParams['font.sans-serif'] = ['SimHei'] 
# plt.rcParams['axes.unicode_minus'] = False

def plot_case_profile(case_name, cases_file='data/cases.csv', summary_file='data/summary_0_499.csv'):
    """
    Load and plot the concentration profile for a given Case.
    Quick sanity check that the raw data behaves as expected.
    
    Args:
        case_name (str): Target Case ID (e.g. 'dp110')
        cases_file (str): Path to the cases file (V_in, C_in, Area)
        summary_file (str): Path to the summary file (concentration profile)
    """
    try:
        # 1. Load data
        if not os.path.exists(cases_file) or not os.path.exists(summary_file):
            print(f"ERROR: cannot find {cases_file} or {summary_file}")
            return

        df_cases = pd.read_csv(cases_file)
        df_summary = pd.read_csv(summary_file)
        
        # 2. Check that the case exists
        if case_name not in df_cases['Case'].values:
            print(f"ERROR: case not found '{case_name}'")
            return
            
        # 3. Extract inlet conditions
        # Grab the per-case C_in, V_in and Area
        case_info = df_cases[df_cases['Case'] == case_name].iloc[0]
        c_in = case_info['C_in']
        v_in = case_info['V_in']
        area = case_info['Area']
        
        # 4. Extract the concentration profile
        # Locate the matching row in the summary table
        # drop(columns=['Case']) Drop non-numeric columns; the remaining column names are distance points ('0', '10', ...)
        profile_row = df_summary[df_summary['Case'] == case_name].drop(columns=['Case']).iloc[0]
        
        # Prepare plot data
        # Cast column names to floats for the X axis (distance)
        distances = [float(col) for col in profile_row.index]
        # Cast row values to floats for the Y axis (concentration)
        concentrations = profile_row.values.astype(float)
        
        # 5. Plotting
        plt.figure(figsize=(12, 6))
        
        # Plot the concentration profile (solid blue)
        plt.plot(distances, concentrations, label=f'Concentration Profile ({case_name})', 
                 color='blue', linewidth=2, marker='.', markersize=5)
        
        # Plot the inlet-concentration reference line (dashed red)
        # Gives a visual sense of decay relative to the inlet concentration
        plt.axhline(y=c_in, color='red', linestyle='--', label=f'Inlet C_in = {c_in:.2e}')
        
        # Labels and title
        plt.title(f"Tunnel Concentration Profile: {case_name}\n"
                  f"(Vin={v_in:.2f} m/s, Area={area:.1f} m^2)", fontsize=14)
        plt.xlabel("Distance from Inlet (m)", fontsize=12)
        plt.ylabel("Particle Concentration (kg/m^3)", fontsize=12)
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3) # Add a grid for easier reading
        
        # 6. Print statistics
        print(f"--- {case_name} Data summary ---")
        print(f"Inlet conditions: C_in={c_in:.2e}, V_in={v_in:.2f}, Area={area:.1f}")
        print(f"Max concentration along path: {concentrations.max():.2e}")
        print(f"Min concentration along path: {concentrations.min():.2e}")
        print(f"Outlet (1100 m) concentration: {concentrations[-1]:.2e}")
        
        plt.show()
        
    except Exception as e:
        print(f"Error: {e}")

# ==========================================
# 6. Run the plot
# ==========================================
# Change the case name you want to inspect here (e.g. 'dp0', 'dp1', 'dp110')
target_case = 'dp110' 
plot_case_profile(target_case)